In [6]:
import truststore
truststore.inject_into_ssl()

import os

from dotenv import load_dotenv
from pinecone import Pinecone

from langchain_community.retrievers import (
    PineconeHybridSearchRetriever
)

from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder


In [7]:
# --------------------------------------------------
# 1. Load environment variables
# --------------------------------------------------

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY is missing from .env")

In [8]:
# --------------------------------------------------
# 2. Initialize Pinecone
# --------------------------------------------------

pc = Pinecone(api_key=PINECONE_API_KEY)

INDEX_NAME = "hybrid-search-langchain"

existing_indexes = [
    index.name for index in pc.indexes.list()
]

if INDEX_NAME not in existing_indexes:

    pc.indexes.create(
        name=INDEX_NAME,
        dimension=384, ##The Pinecone index dimension must match the embedding model output dimension.sentence-transformers/all-MiniLM-L6-v2 generates 384 dimensions
        metric="dotproduct", ##the dot product allows us to combine weighted dense and sparse vectors into a single ranking score.
        deployment={
            "deployment_type": "managed",
            "cloud": "aws",
            "region": "us-east-1"
        }
    )

    print("Pinecone index created successfully!")

else:
    print("Pinecone index already exists!")


# Connect to the index
index = pc.index(name=INDEX_NAME) ##We use index for inserting and searching vectors.



Pinecone index created successfully!


In [13]:
class PineconeIndexAdapter:

    def __init__(self, pinecone_index):
        self.pinecone_index = pinecone_index

    def upsert(self, vectors, **kwargs):
        return self.pinecone_index.upsert(
            vectors=vectors,
            **kwargs
        )

    def __getattr__(self, name):
        return getattr(self.pinecone_index, name)

In [14]:
compatible_index = PineconeIndexAdapter(index)

In [15]:
# --------------------------------------------------
# 3. Initialize dense embedding model
# --------------------------------------------------

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14051.96it/s]


In [17]:
# --------------------------------------------------
# 4. Initialize sparse encoder (BM25)
# --------------------------------------------------

documents = [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"
]

bm25_encoder = BM25Encoder()

# Fit BM25 using the document corpus
bm25_encoder.fit(documents) ##It analyzes the corpus and calculates statistical information such as:Term frequency,Document frequency,Corpus-level statistics,Document-length normalization parameters

# Save the fitted BM25 parameters
bm25_encoder.dump("bm25_values.json") ##why save to json , beacuase We don't want to recalculate the BM25 statistics every time our application starts

# Load the saved parameters
bm25_encoder = BM25Encoder().load("bm25_values.json") ##We reconstruct the encoder from the saved configuration.



100%|██████████| 3/3 [00:00<00:00, 2089.14it/s]

In [ ]:
# --------------------------------------------------
# 5. Initialize hybrid retriever
# --------------------------------------------------
## We store both dense and sparse representations of the same document,
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings, ##Generates dense vectors
    sparse_encoder=bm25_encoder, ##Generates BM25 sparse vectors
    index=compatible_index, ##Pinecone index connection
    alpha=0.5, ##Controls dense vs sparse contribution
    top_k=3, ##Number of documents to retrieve
    namespace="travel-documents" ##Logical partition inside the index
)

In [25]:
# --------------------------------------------------
# 6. Insert documents into Pinecone
# --------------------------------------------------

retriever.add_texts(
    documents,
    namespace="travel-documents"
)

print("Documents inserted successfully!")

100%|██████████| 1/1 [00:01<00:00,  1.89s/it]

Documents inserted successfully!


In [26]:
stats = index.describe_index_stats()

print(stats)

DescribeIndexStatsResponse(dimension=384, total_vector_count=6, metric='dotproduct', namespaces=2)


In [27]:
results = retriever.invoke(
    "What was the first city I visited?"
)

results

[Document(metadata={'score': 0.263601899}, page_content='In 2021, I visited New Orleans'),
 Document(metadata={'score': 0.246669427}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.236530989}, page_content='In 2023, I visited Paris')]

In [28]:
for doc in results:
    print(doc.page_content)
    print(doc.metadata)

In 2021, I visited New Orleans
{'score': 0.263601899}
In 2022, I visited New York
{'score': 0.246669427}
In 2023, I visited Paris
{'score': 0.236530989}
